# AE 352 — Quadrotor Digital Twin
## Software Architecture / Preliminary-Design Notebook

**Project Specification:** *AE 352 Team Project Fall 2026*

**Implementation Philosophy:** keep the physical model, input schedule, numerical solver, and plots separate. Start with NumPy + SciPy + Matplotlib. Condynsate or richer 3D animation can be added later, but this notebook does not depend on it.

**Important:** The plots of *references* in this notebook show prescribed desired paths, **not** physically simulated drone flight and **not** proof of satisfying any performance requirement.

## 0. Requirements and model assumptions

Required flight demonstrations:

1. Hover at **1 m** for **120 s**.
2. Circle of **2 m radius**, **1 m altitude**, **0.5 m/s speed**, for **at least 60 s**.
3. Take off to 1 m, travel 5 m in a straight line at **average 1 m/s**, stop/hover, yaw **-90 degrees** (left in the down-positive, right-handed frame), travel a second 5 m straight segment, stop/hover, and land at **≤ 0.01 m/s** vertical speed.

Project engineering constraints:

- **Six configuration DOF:** three CoM positions plus three attitude angles.
- **Inputs:** four motor torques; rotor angular speeds respond to those torques. **Do not use lift as a directly controllable input** unless it is computed from rotor-speed dynamics.
- Include mass and inertia, gravity, rotor lift, a battery/energy feasibility calculation, and realistic physical parameters.
- Flight plots and results must come from numerical integration of the actual model.

**Proposed assumptions pending team agreement:** rigid four-arm symmetric body; initially no wind/air drag; symmetric inertia tensor if design supports it. The team has adopted right-handed, down-positive inertial and body frames. Gravity acts along inertial +z; rotor thrust acts along body −z. The full drone geometry, rotor numbering, and other modeling assumptions are still pending final approval. These are modeling proposals, *not established project facts*.

## 1. Libraries and Numerical Conventions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from pathlib import Path
from scipy.integrate import solve_ivp

np.set_printoptions(precision=5, suppress=True)

G = 9.81                 # m/s^2, gravity toward inertial +z
DEG = 180.0 / np.pi
RAD = np.pi / 180.0

def rps_to_rad_s(rotations_per_second):
    return 2.0 * np.pi * np.asarray(rotations_per_second) # can use an array of four rotor speeds if we want

def rad_s_to_rps(angular_velocity_rad_s):
    return np.asarray(angular_velocity_rad_s) / (2.0 * np.pi)

def altitude_from_z(z):
    """Altitude above the z=0 ground plane."""
    return -np.asarray(z)

## 2. Physical Parameters

**Fill actual values once teammates document components.**

| Parameter | Units | Responsible party / reason |
|---|---|---|
| `mass_kg` | kg | Total vehicle mass, including battery |
| `arm_length_m` | m | CoM to each rotor axis |
| `inertia_kg_m2` | kg·m² | 3×3 inertia tensor about CoM in body frame |
| `kT` | N·s²/rad² | Propeller thrust model coefficient for rad/s |
| `kQ` | N·m·s²/rad² | Aerodynamic rotor resistance / moment coefficient |
| `rotor_inertia_kg_m2` | kg·m² | Rotor rotational inertia |
| `max_rotor_speed_rad_s` | rad/s | Motor+propeller safe limit |
| `max_motor_torque_Nm` | N·m | Torque limit per motor |
| `battery_capacity_Wh` | Wh | Usable energy, not merely advertised capacity |
| `motor_system_efficiency` | dimensionless | Preliminary power conversion model |

If the team derives propeller forces using disk-loading or another model instead of `kT * omega**2`, replace the thrust function rather than the entire notebook.

In [ ]:
@dataclass
class DroneParameters:
    mass_kg: float | None = None
    arm_length_m: float | None = None
    inertia_kg_m2: np.ndarray | None = None
    kT: float | None = None
    kQ: float | None = None
    rotor_inertia_kg_m2: float | None = None
    max_rotor_speed_rad_s: float | None = None
    max_motor_torque_Nm: float | None = None
    battery_capacity_Wh: float | None = None
    motor_system_efficiency: float | None = None
    gravity_m_s2: float = G

p = DroneParameters() # fill


def missing_parameters(params):
    """Returns the names of missing parameters"""
    return [name for name in (
        "mass_kg", "arm_length_m", "inertia_kg_m2", "kT", "kQ",
        "rotor_inertia_kg_m2", "max_rotor_speed_rad_s", "max_motor_torque_Nm",
        "battery_capacity_Wh", "motor_system_efficiency"
    ) if getattr(params, name) is None]


def validate_parameters(params):
    """Determines if entered parameters are acceptable for simulation"""
    missing = missing_parameters(params)
    if missing:
        raise ValueError(f"Real physical inputs are missing: {', '.join(missing)}")
    I = np.asarray(params.inertia_kg_m2, dtype=float)
    assert I.shape == (3, 3), "Inertia tensor must be 3x3"
    assert np.allclose(I, I.T), "Inertia tensor must be symmetric"
    assert np.all(np.linalg.eigvalsh(I) > 0), "Inertia must be positive-definite"
    assert 0 < params.mass_kg and 0 < params.arm_length_m
    assert 0 < params.kT and 0 < params.kQ
    assert 0 < params.rotor_inertia_kg_m2
    assert 0 < params.max_rotor_speed_rad_s and 0 < params.max_motor_torque_Nm
    assert 0 < params.battery_capacity_Wh and 0 < params.motor_system_efficiency <= 1
    assert 0.1 <= params.mass_kg <= 10.0, "Project mass must be between 0.1 and 10 kg"
    return True

print("Awaiting team inputs:", missing_parameters(p))

## 3. Reference Frames and Transport Theorem

**Team coordinate convention:**

- Inertial frame I: +x forward, +y right, +z downward. Gravity acts along +z.
- Body frame B: origin at the drone's center of mass; +x toward the front, +y toward the right, +z downward, opposite rotor thrust.
- Both frames are right-handed.
- Euler angles use the 3–2–1 sequence: `R_BI = Rz(yaw) @ Ry(pitch) @ Rx(roll)`.
- `R_BI` transforms body-frame vector components into inertial-frame components.
- Positive yaw is a right turn; negative yaw is a left turn.
- Body angular velocity is `[p, q, r]`, which is not generally equal to `[roll_dot, pitch_dot, yaw_dot]`.

The **transport theorem** links inertial and body vector derivatives:

$$\left(\frac{d \mathbf A}{dt}\right)_I = \left(\frac{d \mathbf A}{dt}\right)_B + \boldsymbol\omega \times \mathbf A$$

For body angular momentum $\mathbf H_B = \mathbf J_B\boldsymbol\omega_B$, it produces the gyroscopic $\boldsymbol\omega_B\times(\mathbf J_B\boldsymbol\omega_B)$ term. This is the reason the rotation subsystem is naturally computed in a rotating body-fixed frame.

**Do not mix** 3–1–3 Euler-angle formulas from Chapter 6 with the 3–2–1 convention used here.

In [ ]:
def R_body_to_inertial(roll, pitch, yaw):
    """Constructs rotation matrix that transforms vectors from from the body frame to the inertial frame"""
    cr, sr = np.cos(roll), np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw), np.sin(yaw)
    Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])
    Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])
    Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])
    return Rz @ Ry @ Rx


def euler_rates_321(rpy, body_rates): # black box atm
    roll, pitch, yaw = rpy
    p_rate, q_rate, r_rate = body_rates
    if abs(np.cos(pitch)) < 1e-6:
        raise ValueError("Euler 3-2-1 kinematic singularity; consider quaternions")
    sr, cr = np.sin(roll), np.cos(roll)
    ct, tt = np.cos(pitch), np.tan(pitch)
    mapping = np.array([
        [1, sr * tt, cr * tt],
        [0, cr, -sr],
        [0, sr / ct, cr / ct],
    ])
    return mapping @ np.array([p_rate, q_rate, r_rate])

# Independent unit tests that do not need the vehicle EOM:
assert np.allclose(R_body_to_inertial(0, 0, 0), np.eye(3))
for a, b, c in [(0, 0, 0), (0.1, -0.2, 0.3), (0.4, 0.1, -0.8)]:
    R = R_body_to_inertial(a, b, c)
    assert np.allclose(R.T @ R, np.eye(3))
    assert np.isclose(np.linalg.det(R), 1.0)
assert np.allclose(euler_rates_321(np.zeros(3), np.array([1, 2, 3])), [1, 2, 3])
print("Rotation-matrix and zero-attitude kinematics tests passed")

## 4. Numerical State Definition

The drone has **6 mechanical degrees of freedom** but more than six first-order **state variables**:

$$
\mathbf y=[x,y,z,\;v_x,v_y,v_z,\;\phi,\theta,\psi,\;p,q,r,\;\Omega_1,\Omega_2,\Omega_3,\Omega_4,\;E_{\rm used}]^T.
$$

- First 12 quantities: 6-DOF rigid-body dynamics, positions and rates.
- Four $\Omega_i$: rotor angular velocities [rad/s], whose derivatives depend on motor torques.
- Last state: cumulative battery energy used [J].

**17 first-order states**, not 17 mechanical degrees of freedom

In [ ]:
# Shortcuts to access state vector
POS = slice(0, 3)
VEL = slice(3, 6)
RPY = slice(6, 9)
BODY_RATES = slice(9, 12)
ROTOR_SPEEDS = slice(12, 16)
ENERGY_USED = 16
N_STATES = 17


def initial_state(position=(0., 0., -1.), velocity=(0., 0., 0.),
                  rpy=(0., 0., 0.), body_rates=(0., 0., 0.),
                  rotor_speeds=(0., 0., 0., 0.), energy_used=0.):
    """Sets the initial state of the drone"""
    x0 = np.zeros(N_STATES)
    x0[POS] = position
    x0[VEL] = velocity
    x0[RPY] = rpy
    x0[BODY_RATES] = body_rates
    x0[ROTOR_SPEEDS] = rotor_speeds
    x0[ENERGY_USED] = energy_used
    return x0

x0 = initial_state()
print("States:", x0.shape, "\n", x0)

## 5. Rotor Positions, Thrust, and Hover Sanity Check

With **+ configuration** and the body axes above, the rotor positions measured from the CoM are $(L,0,0)$, $(0,L,0)$, $(-L,0,0)$, and $(0,-L,0)$.

Simplified thrust law assuming constant thrust and aerodynamic torque coefficients (follows propeller-load approach from guest lecture page 11):

$$T_i = k_T\Omega_i^2,\qquad k_T=\frac{1}{2} \rho AR^2C_T$$

$$Q_i = k_Q\Omega_i^2,\qquad k_Q=\frac{1}{2} \rho AR^3C_P$$

Here, $\Omega_i$ is the nonnegative rotor speed in rad/s, $T_i$ is thrust magnitude in N, and $Q_i$ is aerodynamic resistance torque magnitude in Nm

The coefficients $k_T$ and $k_Q$ must be obtained from documented propeller data or a justified physical estimate. The model neglects changes in aerodynamic coefficients with flight velocity and inflow

Since body +z points downward, the thrust force on the drone is:

$$\mathbf F_{i,B}=[0,0,-T_i]^T$$

### Hover Equilibrium

For steady, level hover:

$$
\sum_{i=1}^4 T_i = mg.
$$

For four identical rotors:

$$
\Omega_{\mathrm{hover}} = \sqrt{\frac{mg}{4k_T}}.
$$

At steady rotor speed, each motor must supply an aerodynamic holding torque of

$$
u_{\mathrm{hover}} = k_Q\Omega_{\mathrm{hover}}^2.
$$

**Rotor spin directions and the final vehicle configuration must be confirmed by the design team.**

The vehicle pitch/roll torques include $\sum_i\mathbf r_i\times\mathbf F_i$. **Yaw torque needs special care:** motor-reaction torque, air-resistance torque, and rotor angular momentum must not be double-counted. Finalize that coupling with the EOM team before simulation.

In [ ]:
# Provisional spin directions:
# +1 = rotation about body +z (clockwise viewed from above)
# -1 = rotation about body -z (counterclockwise viewed from above)
SPIN_DIRECTIONS = np.array([1, -1, 1, -1])


def rotor_positions_B(arm_length_m):
    """Rotor positions relative to CoM, in body coordinates."""
    L = float(arm_length_m)
    return np.array([
        [ L,  0., 0.],  # 1: front
        [0., -L,  0.],  # 2: left
        [-L,  0., 0.],  # 3: rear
        [0.,  L,  0.]   # 4: right
    ])


def thrust_per_rotor(rotor_speeds_rad_s, params):
    """Positive thrust magnitudes, in N."""
    if params.kT is None:
        raise ValueError("Set propeller thrust coefficient kT")

    speeds = np.asarray(rotor_speeds_rad_s, dtype=float)

    if speeds.shape != (4,) or np.any(speeds < 0):
        raise ValueError("Expected four nonnegative rotor speeds")

    return params.kT * speeds**2


def aerodynamic_torque_per_rotor(rotor_speeds_rad_s, params):
    """Positive aerodynamic resistance torque magnitudes, in N m."""
    if params.kQ is None:
        raise ValueError("Set propeller torque coefficient kQ")

    speeds = np.asarray(rotor_speeds_rad_s, dtype=float)

    if speeds.shape != (4,) or np.any(speeds < 0):
        raise ValueError("Expected four nonnegative rotor speeds")

    return params.kQ * speeds**2


def estimate_hover(params):
    """Steady level-hover estimate, not a flight simulation."""
    if params.mass_kg is None or params.kT is None or params.kQ is None:
        raise ValueError("Hover estimate requires mass, kT, and kQ")

    rotor_speed = np.sqrt(
        params.mass_kg * params.gravity_m_s2 / (4.0 * params.kT)
    )

    holding_torque = params.kQ * rotor_speed**2
    mechanical_power_total = 4.0 * holding_torque * rotor_speed

    return dict(
        rotor_speed_rad_s=rotor_speed,
        rotor_speed_rps=rad_s_to_rps(rotor_speed),
        thrust_N_each=params.mass_kg * params.gravity_m_s2 / 4.0,
        motor_holding_torque_Nm_each=holding_torque,
        mechanical_power_W_total=mechanical_power_total
    )


# Geometry sanity checks; no physical parameters required
r_test = rotor_positions_B(1.0)
F_test = np.tile([0., 0., -1.], (4, 1))

assert np.allclose(
    np.sum(np.cross(r_test, F_test), axis=0),
    np.zeros(3)
)
assert np.sum(SPIN_DIRECTIONS) == 0

print("Rotor geometry checks passed.")
print("Awaiting approved geometry, kT, kQ, and physical parameters.")

## 6. MOTOR TORQUES and Rotor Differential Equations — **Implemented (canidate model)**

Motor torque must overcome the torque resisting rotor rotation before the rotor can accelerate

Each motor receives a commanded shaft torque $u_i(t)$ [Nm]. A candidate reduced rotor-spin equation is:

$$J_r\dot\Omega_i=u_i-Q_{\mathrm{aero},i},\qquad Q_{\mathrm{aero},i} \approx Q_i = k_Q\Omega_i^2$$

So, $$\dot\Omega_i = \frac{u_i - k_Q\Omega_i^2}{J_r}, \qquad i=1,2,3,4$$

The rotor acceleration should be returned as four derivatives $\dot\Omega_i$

For constant rotor speed:

$$
\dot{\Omega}_i=0
$$

Therefore:

$$
u_i=k_Q\Omega_i^2
$$

For steady, symmetric hover, all four rotors have equal speed and holding torque

**Group must confirm:** aerodynamic torque model, rotor inertia, whether body yaw dynamics affect rotor angular momentum, allowed positive/negative motor torque, and rotor speed limits. Do not substitute instantaneous speed commands for torque commands without stating an approximation.

In [ ]:
def motor_angular_accelerations(t, state, motor_torques_Nm, params):
    """Calculate the angular acceleration of each rotor [rad/s^2]"""

    # Current rotor speeds (rad/s)
    omega = np.asarray(state[ROTOR_SPEEDS], dtype=float)

    # Applied motor shaft torques (N m)
    u = np.asarray(motor_torques_Nm, dtype=float)

    # Validate inputs
    if u.shape != (4,):
        raise ValueError("Expected four motor torques")

    if not np.all(np.isfinite(u)) or not np.all(np.isfinite(omega)):
        raise ValueError("Motor torques and rotor speeds must be finite")

    if np.any(u < 0):
        raise ValueError("Negative motor torque is not supported")

    if np.any(omega < 0):
        raise ValueError("Rotor speeds must be nonnegative")

    if params.rotor_inertia_kg_m2 is None or params.rotor_inertia_kg_m2 <= 0:
        raise ValueError("Set a positive rotor rotational inertia")

    if params.max_motor_torque_Nm is not None:
        if np.any(u > params.max_motor_torque_Nm):
            raise ValueError("Motor torque exceeds selected motor limit")

    # Aerodynamic resistance torques (N m)
    Q_aero = aerodynamic_torque_per_rotor(omega, params)

    # Newton's rotational equation: Jr * omega_dot = u - Q_aero
    omega_dot = (u - Q_aero) / params.rotor_inertia_kg_m2

    return omega_dot

In [ ]:
# Section 6 verification
# Illustrative values ONLY — not approved physical parameters

test_params = DroneParameters(
    kQ=1e-8,
    rotor_inertia_kg_m2=1e-5,
    max_motor_torque_Nm=0.02
)

test_state = initial_state(
    rotor_speeds=(500., 500., 500., 500.)
)

# Torque required to maintain constant rotor speed
u_hold = aerodynamic_torque_per_rotor(
    test_state[ROTOR_SPEEDS], test_params
)

# Test 1: Holding torque -> zero angular acceleration
acc_hold = motor_angular_accelerations(
    0., test_state, u_hold, test_params
)

assert np.allclose(acc_hold, 0.)

# Test 2: More motor torque -> rotors accelerate
acc_up = motor_angular_accelerations(
    0., test_state, 1.2 * u_hold, test_params
)

assert np.all(acc_up > 0.)

# Test 3: Less motor torque -> rotors decelerate
acc_down = motor_angular_accelerations(
    0., test_state, 0.8 * u_hold, test_params
)

assert np.all(acc_down < 0.)

print("Section 6 motor dynamics tests passed.")
print("Holding:", acc_hold)
print("Accelerating:", acc_up)
print("Decelerating:", acc_down)

## 7. Rotor Forces and Body Moments — **EOM TODO 2**

Expected outputs:

- Total applied **body-frame force** $\mathbf F_B$ [N] from the four rotors (exclude gravity, which is applied in the inertial frame below).
- Total applied **body-frame moment about the CoM** $\boldsymbol\tau_B$ [N m].

Typical force moment contribution: $\sum_i \mathbf r_i\times \mathbf F_{i,B}$, with a separately derived and consistently signed yaw/rotor-reaction contribution. Rotor gyroscopic coupling may need to be added to the rotational EOM or moments, **but not twice**.

In [ ]:
def rotor_forces_and_body_moments(t, state, motor_torques_Nm,
                                  rotor_accelerations, params):
    """Converts rotor behavior into forces and moments on drone"""
    """TODO 2: return (force_body_3vector, moment_body_3vector)."""
    raise NotImplementedError("TODO 2: team-approved force/moment allocation is not inserted yet")

## 8. Center-of-Mass Translational EOM — **Implemented**

Candidate inertial-frame force balance for the baseline model:

$$m\ddot{\mathbf r}_I=\mathbf R_{BI}\mathbf F_{B}+\begin{bmatrix}0\\0\\mg\end{bmatrix}$$

This model needs **no added Coriolis force on the CoM** because CoM position and velocity are kept in the **inertial frame**. More complicated terms are necessary if a *different* body-frame translational-state convention is used.

In [ ]:
def translational_acceleration_I(t, state, total_force_B, params):
    """Determines acceleration of CoM"""
    # CoM acceleration in the down-positive inertial frame

    R = R_body_to_inertial(*state[RPY])

    force_B = np.asarray(total_force_B, dtype=float)

    gravity_I = np.array([0., 0., params.gravity_m_s2])

    return R @ force_B / params.mass_kg + gravity_I

In [ ]:
# Independent translational EOM sanity checks
test_params = DroneParameters(mass_kg=1.0)
test_state = initial_state()

# Level hover: thrust equals weight
F_hover = np.array([0., 0., -9.81])

a_hover = translational_acceleration_I(
    0., test_state, F_hover, test_params
)

assert np.allclose(a_hover, [0., 0., 0.])

# More thrust than weight: acceleration must be UP (-z)
F_climb = np.array([0., 0., -10.81])

a_climb = translational_acceleration_I(
    0., test_state, F_climb, test_params
)

assert np.allclose(a_climb, [0., 0., -1.])

print("Down-positive translational EOM checks passed")

## 9. Rotational Dynamics / Transport Theorem — **EOM TODO 4**

For a rigid body with a constant body-fixed inertia tensor $\mathbf J_B$, the familiar baseline form is:

$$\boldsymbol\tau_B=\mathbf J_B\dot{\boldsymbol\omega}_B+\boldsymbol\omega_B\times(\mathbf J_B\boldsymbol\omega_B).$$

Equivalently,

$$\dot{\boldsymbol\omega}_B=\mathbf J_B^{-1}\left[\boldsymbol\tau_B-\boldsymbol\omega_B\times(\mathbf J_B\boldsymbol\omega_B)\right].$$

**Do not treat this as the approved full rotor/body EOM** until rotor spinning angular momentum and actuator reaction moments have been accounted for consistently. Chapter 6 discusses the inertial/body transformation and Euler's rotational equations. Attitude kinematics are then integrated through `euler_rates_321()` separately.

In [ ]:
def rotational_acceleration_B(t, state, total_moment_B, params):
    """Calculates angular acceleration based on current state"""
    """TODO 4: return body angular acceleration [p_dot, q_dot, r_dot]."""
    # Candidate bare-body formula, only after the rotor coupling convention is agreed:
    # omega = state[BODY_RATES]
    # J = np.asarray(params.inertia_kg_m2, dtype=float)
    # return np.linalg.solve(J, total_moment_B - np.cross(omega, J @ omega))
    raise NotImplementedError("TODO 4: finalize rigid-body/rotor coupled rotational EOM")

## 10. Battery Power Model — **MODEL TODO 5**

Battery constraint is **required**, not an optional add-on. We accumulate electrical energy consumption by integrating $\dot E_{\rm used}=P_{\rm electric}$.

For a first estimate, the sum of positive mechanical shaft powers may be divided by a documented overall efficiency, but motoring/braking behavior, electronics loads, and usable battery capacity must be defined. **Do not treat a toy efficiency assumption as a documented battery result**.

$$E_{\rm capacity}[\mathrm J]=3600\,E_{\rm capacity}[\mathrm{Wh}]$$

In [ ]:
def electrical_power_W(t, state, motor_torques_Nm, params):
    """Instantaneous electrical power draw"""
    """TODO 5: nonnegative electrical power draw [W]."""
    raise NotImplementedError("TODO 5: insert the approved motor/battery power model")


def remaining_energy_J(energy_used_J, battery_capacity_Wh):
    return (3600.0*battery_capacity_Wh) - energy_used_J

## 11. Motor Command Schedule — **INPUT TODO 6**

The **input is four motor torques**, not a direct 3D force/attitude trajectory. Start with prescribed piecewise-smooth motor torque schedules and an analytical steady-hover equilibrium. Will most likely need more schedules.

In [ ]:
def motor_torque_schedule(t, state, params):
    """Determines how much torque each motor should apply at the current time and state"""
    """TODO 6: return four motor torques [N m] appropriate to the chosen mission."""
    raise NotImplementedError("TODO 6: define an actual motor-torque schedule")

## 12. ONE numerical ODE interface — connect the approved equations here

The physical model is **one coupled right-hand side** `state_dot = drone_rhs(t, state, params, schedule)`.

Dataflow: command -> four rotor accelerations -> rotor forces/moments -> CoM and attitude accelerations -> energy draw -> first-order state derivative. The only reason to edit `drone_rhs()` is if the state definition changes.

In [ ]:
def drone_rhs(t, state, params, schedule):
    """Combines all physics functions into one vector of state derivatives for the numerical solver"""
    u = np.asarray(schedule(t, state, params), dtype=float)
    if u.shape != (4,):
        raise ValueError("Motor command must have exactly four torques")

    rotor_ddot = motor_angular_accelerations(t, state, u, params)
    force_B, moment_B = rotor_forces_and_body_moments(t, state, u, rotor_ddot, params)
    acc_I = translational_acceleration_I(t, state, force_B, params)
    omega_dot_B = rotational_acceleration_B(t, state, moment_B, params)
    energy_dot = electrical_power_W(t, state, u, params)

    state_dot = np.zeros(N_STATES)
    state_dot[POS] = state[VEL]
    state_dot[VEL] = acc_I
    state_dot[RPY] = euler_rates_321(state[RPY], state[BODY_RATES])
    state_dot[BODY_RATES] = omega_dot_B
    state_dot[ROTOR_SPEEDS] = rotor_ddot
    state_dot[ENERGY_USED] = energy_dot
    return state_dot

# Turn this on only AFTER all model TODOs are replaced and physically checked!!!
EOM_APPROVED_AND_IMPLEMENTED = False

## 13. Numerical Integration with SciPy — ready once EOM is filled

Use `solve_ivp()` to integrate the state rather than separately integrating x, angles, and rotor RPM. That keeps the simulation time history and coupling consistent.

When running a real flight, use separate schedules and initial conditions for the three missions. Handle the ground with an event/termination condition; do not let an unmodeled vehicle freely pass below ground and claim that it landed.

In [ ]:
def simulate(params, initial_conditions, t_final_s, schedule=motor_torque_schedule,
             samples=1201, max_step_s=0.05):
    validate_parameters(params)
    t_eval = np.linspace(0.0, t_final_s, samples)
    sol = solve_ivp(
        fun=lambda t, y: drone_rhs(t, y, params, schedule),
        t_span=(0.0, float(t_final_s)),
        y0=np.asarray(initial_conditions, dtype=float),
        t_eval=t_eval,
        rtol=1e-7, atol=1e-9, max_step=max_step_s,
    )
    if not sol.success:
        raise RuntimeError(f"ODE solver failed: {sol.message}")
    return sol

if not EOM_APPROVED_AND_IMPLEMENTED:
    print("Physical flight simulation SKIPPED: rotor/body EOM, motor inputs and power remain TODOs.")

## 14. Three Target Trajectories — safe to build now

These show **what the model must achieve**. They are reference data, not model output. Keep the plotted desired path distinct from the eventual `solve_ivp` path.

In [ ]:
def reference_hover(times_s):
    t = np.asarray(times_s)
    return np.column_stack([np.zeros_like(t), np.zeros_like(t), -np.ones_like(t)])


def reference_circle(times_s, radius_m=2.0, altitude_m=1.0, speed_m_s=0.5):
    t = np.asarray(times_s)
    w = speed_m_s / radius_m
    return np.column_stack([
        radius_m * np.cos(w * t),
        radius_m * np.sin(w * t),
        np.full_like(t, -altitude_m),
    ])

# High-level maneuver specification, not a dynamically feasible control schedule.
MISSION3_WAYPOINTS = [
    {"phase":"takeoff",       "x_m":0., "y_m":0.,  "z_m":0.,  "yaw_deg":0.},
    {"phase":"reach 1 m",     "x_m":0., "y_m":0.,  "z_m":-1., "yaw_deg":0.},
    {"phase":"5 m leg 1",     "x_m":5., "y_m":0.,  "z_m":-1., "yaw_deg":0.},
    {"phase":"stop/hover",    "x_m":5., "y_m":0.,  "z_m":-1., "yaw_deg":0.},
    {"phase":"yaw left 90°",  "x_m":5., "y_m":0.,  "z_m":-1., "yaw_deg":-90.},
    {"phase":"5 m leg 2",     "x_m":5., "y_m":-5., "z_m":-1., "yaw_deg":-90.},
    {"phase":"stop/hover",    "x_m":5., "y_m":-5., "z_m":-1., "yaw_deg":-90.},
    {"phase":"touchdown",     "x_m":5., "y_m":-5., "z_m":0.,  "yaw_deg":-90.},
]

# Keep 5 m / 1 m/s = 5 seconds as the average-travel-time target for EACH horizontal leg.
# Landing maximum vertical touchdown speed: 0.01 m/s. Do not confuse this with average descent speed.

t_h = np.linspace(0, 120, 501)
t_c = np.linspace(0, 60, 501)
hover_ref = reference_hover(t_h)
circle_ref = reference_circle(t_c)
print(f"Circular path: angular rate = {0.5/2:.3f} rad/s, one lap = {2*np.pi*2/0.5:.2f} s")
print(f"Circle duration = 60 s, approximate laps = {60/(2*np.pi*2/0.5):.2f}")

## 15. Plot Mission REFERENCES (not simulated flight)

Run these immediately to check frame orientations, horizontal directions, the circle definition, and the stop/yaw/land order before the equations arrive.

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Circle reference
ax.plot(
    circle_ref[:, 0],
    circle_ref[:, 1],
    altitude_from_z(circle_ref[:, 2]),
    label='Circle reference — NOT simulated'
)

# Hover reference
ax.scatter(
    hover_ref[0, 0],
    hover_ref[0, 1],
    altitude_from_z(hover_ref[0, 2]),
    label='Hover reference',
    s=45
)

# Mission 3 waypoints
mission_xyz = np.array([
    [wp['x_m'], wp['y_m'], wp['z_m']]
    for wp in MISSION3_WAYPOINTS
])

ax.plot(
    mission_xyz[:, 0],
    mission_xyz[:, 1],
    altitude_from_z(mission_xyz[:, 2]),
    'o--',
    label='Maneuver waypoints — NOT simulated'
)

ax.set(
    xlabel='x [m]',
    ylabel='y [m]',
    zlabel='Altitude [m]',
    title='Required reference geometry (not model results)'
)

ax.set_zlim(bottom=0)
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 16. Visualization of Actual Integrated States — ready now (eh)

This function will plot the actual CoM position (3 DOF) and actual orientation (3 DOF), which are specifically appropriate for the team report. Call it only on an **actual numerical ODE solution**.

In [ ]:
def plot_actual_6dof(solution, title='Simulated drone 6-DOF response'):
    t = solution.t
    y = solution.y

    fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)

    names = [
        'x [m]', 'y [m]', 'Altitude [m]',
        'roll [deg]', 'pitch [deg]', 'yaw [deg]'
    ]

    # Translational position
    for j in range(3):
        values = altitude_from_z(y[j]) if j == 2 else y[j]

        axes[0, j].plot(t, values)
        axes[0, j].set_ylabel(names[j])
        axes[0, j].grid(True)

    # Rotational orientation
    for j in range(3):
        axes[1, j].plot(t, y[6 + j] * DEG)
        axes[1, j].set_ylabel(names[3 + j])
        axes[1, j].set_xlabel('time [s]')
        axes[1, j].grid(True)

    fig.suptitle(title)
    fig.tight_layout()
    return fig


def plot_actual_trajectory(solution):
    xyz = solution.y[POS]

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot(
        xyz[0],
        xyz[1],
        altitude_from_z(xyz[2]),
        label='Actual ODE solution'
    )

    ax.set(
        xlabel='x [m]',
        ylabel='y [m]',
        zlabel='Altitude [m]',
        title='Actual integrated 3D trajectory'
    )

    ax.set_zlim(bottom=0)
    ax.legend()
    return fig


def plot_actual_rotors_and_energy(solution):
    t, y = solution.t, solution.y

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    for i in range(4):
        axes[0].plot(
            t,
            rad_s_to_rps(y[12 + i]),
            label=f'rotor {i+1}'
        )

    axes[0].set_ylabel('Rotor speed [rev/s]')
    axes[0].legend(ncol=4)
    axes[0].grid(True)

    axes[1].plot(t, y[ENERGY_USED] / 3600)
    axes[1].set(
        xlabel='time [s]',
        ylabel='Electrical energy used [Wh]'
    )
    axes[1].grid(True)

    fig.tight_layout()
    return fig

## 17. Validation Tests and Mission Scoring

**Mandatory before believing figures:**

- Rotation matrix orthonormal and determinant +1.
- Equal motor/rotor conditions give no unintended pitch or roll; with alternating counter-rotation also verify net yaw moment at steady hover.
- Hover balance: computed total thrust equals $mg$ at the predicted rotor equilibrium.
- Every rotor stays within its allowed speed and motor torque limits.
- Battery energy remains positive and used energy is nondecreasing under the chosen power convention.
- Numerical results do not materially change when time step/tolerances are tightened.
- Verify actual height, average speed, circle radius, yaw change, touchdown speed, and mission duration from **simulated** states, not target trajectory alone.
- Do **not** silently select tolerances (e.g. ±5 cm altitude) and call them professor requirements; list them as team-defined acceptance criteria.

In [ ]:
def get_basic_flight_metrics(solution, params):
    """Convenient numeric outputs; not automatically a pass/fail certificate"""

    t, y = solution.t, solution.y
    xyz, vxyz = y[POS], y[VEL]

    altitude = altitude_from_z(xyz[2])
    speed = np.linalg.norm(vxyz, axis=0)
    rotor_speeds = y[ROTOR_SPEEDS]
    energy_used = y[ENERGY_USED, -1]

    return {
        'duration_s': float(t[-1] - t[0]),
        'min_altitude_m': float(altitude.min()),
        'max_altitude_m': float(altitude.max()),
        'max_speed_m_s': float(speed.max()),
        'max_rotor_speed_rad_s': float(rotor_speeds.max()),
        'battery_energy_used_Wh': float(energy_used / 3600.),
        'battery_energy_remaining_Wh': float(
            params.battery_capacity_Wh - energy_used / 3600.
        ),
        'final_yaw_deg': float(y[RPY][2, -1] * DEG),

        # Signed vertical speed: positive means downward
        'final_vertical_speed_m_s': float(vxyz[2, -1]),
    }


def hover_diagnostics(solution, altitude_ref_m=1.0):
    xyz = solution.y[POS]
    altitude = altitude_from_z(xyz[2])

    return {
        'max_height_error_m': float(
            np.max(np.abs(altitude - altitude_ref_m))
        ),
        'max_horizontal_drift_m': float(
            np.max(np.linalg.norm(xyz[:2] - xyz[:2, [0]], axis=0))
        ),
        'duration_s': float(solution.t[-1] - solution.t[0]),
    }


def circle_diagnostics(solution, radius_m=2.,
                       speed_ref_m_s=0.5, altitude_m=1.):

    xyz, vxyz = solution.y[POS], solution.y[VEL]

    altitude = altitude_from_z(xyz[2])
    actual_radius = np.linalg.norm(xyz[:2], axis=0)
    actual_horizontal_speed = np.linalg.norm(vxyz[:2], axis=0)

    return {
        'max_radius_error_m': float(
            np.max(np.abs(actual_radius - radius_m))
        ),
        'max_speed_error_m_s': float(
            np.max(np.abs(actual_horizontal_speed - speed_ref_m_s))
        ),
        'max_height_error_m': float(
            np.max(np.abs(altitude - altitude_m))
        ),
        'duration_s': float(solution.t[-1] - solution.t[0]),
    }


def mission3_diagnostics(solution, phase_times):
    """TODO: extract two straight-leg durations, 90 deg left yaw,
    hover stops, and touchdown speed."""
    raise NotImplementedError(
        "Define mission phase boundaries and touchdown event after EOM integration"
    )

## 18. Save Report Figures and GitHub Handoff

Repository Idea:

```text
AE352-quadrotor/
├── quadrotor_digital_twin.ipynb
├── README.md
├── requirements.txt
├── figures/
└── references/
```

`requirements.txt` (suggested):

```text
numpy
scipy
matplotlib
jupyter
```

Recommended reported simulation figures: each CoM coordinate vs time, Euler angles vs time, 3D trajectory, four rotor speeds, and battery energy vs time. The project draft requires a mathematical model, results/discussion, team contribution list, and GitHub code link. Preserve this notebook and export tested figures rather than including screenshots of untested reference trajectories as flight results.

In [ ]:
FIGURES_DIR = Path('figures')

def export_figure(fig, filename, dpi=220):
    FIGURES_DIR.mkdir(exist_ok=True)
    output = FIGURES_DIR / filename
    fig.savefig(output, dpi=dpi, bbox_inches='tight')
    print('Saved:', output)
    return output

print('Notebook scaffold complete. Next: add physical parameters and implement model TODOs 1–6.')